In [1]:
# Install required libraries
!pip install nltk numpy

# Download WikiText-2 dataset
import urllib.request
import os

url = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt"
file_name = "wiki.train.txt"

if not os.path.exists(file_name):
    print("Downloading WikiText-2 dataset...")
    urllib.request.urlretrieve(url, file_name)
    print("WikiText-2 downloaded successfully!")
else:
    print("WikiText-2 is already downloaded!")

print("Setup completed!")

WikiText-2 downloaded successfully!
Setup completed!


In [ ]:
import re
import numpy as np
from collections import Counter

with open("wiki.train.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("WikiText-2 dataset loaded successfully!")
print("Total characters:", len(text))

def tokenize(text):
    # Convert text to lowercase
    text = text.lower()

    # Keep alphabets, numbers, spaces and apostrophes
    text = re.sub(r"[^a-z0-9\s']", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Split into words
    tokens = text.split()

    return tokens


tokens = tokenize(text)

print("Total tokens:", len(tokens))
print("First 20 tokens:")
print(tokens[:20])

unigrams = Counter(tokens)

print("\nUnique words:", len(unigrams))

bigrams = Counter()

for i in range(len(tokens) - 1):
    word1 = tokens[i]
    word2 = tokens[i + 1]

    bigrams[(word1, word2)] += 1

print("Unique bigrams:", len(bigrams))

trigrams = Counter()

for i in range(len(tokens) - 2):
    word1 = tokens[i]
    word2 = tokens[i + 1]
    word3 = tokens[i + 2]

    trigrams[(word1, word2, word3)] += 1

print("Unique trigrams:", len(trigrams))

def unigram_probability(word):

    total_words = len(tokens)

    if total_words == 0:
        return 0

    return unigrams[word] / total_words


def bigram_probability(word1, word2):

    word1_count = unigrams[word1]
    bigram_count = bigrams[(word1, word2)]

    if word1_count == 0:
        return 0

    return bigram_count / word1_count


def trigram_probability(word1, word2, word3):

    bigram_count = bigrams[(word1, word2)]
    trigram_count = trigrams[(word1, word2, word3)]

    if bigram_count == 0:
        return 0

    return trigram_count / bigram_count

def predict_next_words(sentence, top_n=5):

    # Tokenize user sentence
    words = tokenize(sentence)

    if len(words) == 0:
        return []

    candidates = {}

    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        for (w1, w2, w3), count in trigrams.items():

            if w1 == word1 and w2 == word2:

                probability = trigram_probability(
                    w1, w2, w3
                )

                candidates[w3] = probability

    last_word = words[-1]

    for (w1, w2), count in bigrams.items():

        if w1 == last_word:

            probability = bigram_probability(
                w1, w2
            )

            if w2 not in candidates:

                candidates[w2] = probability

    if len(candidates) == 0:

        for word, count in unigrams.most_common(50):

            candidates[word] = unigram_probability(word)

    predictions = sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )


    # Return Top-N predictions

    return predictions[:top_n]

def show_predictions(sentence, top_n=5):

    predictions = predict_next_words(
        sentence,
        top_n
    )

    print("\n" + "=" * 55)
    print("Input:", sentence)
    print("=" * 55)

    if len(predictions) == 0:

        print("No predictions found.")

    else:

        print("\nTop predicted next words:\n")

        for i, (word, probability) in enumerate(
            predictions, 1
        ):

            print(
                f"{i}. {word:<15} "
                f"Probability: {probability:.4f}"
            )

show_predictions(
    "machine learning is",
    top_n=5
)

test_sentences = [
    "machine learning",
    "the united",
    "one of the",
    "the first",
    "new york",
    "computer science",
    "artificial intelligence"
]

print("\n\nTESTING DIFFERENT SENTENCES")

for sentence in test_sentences:

    show_predictions(
        sentence,
        top_n=5
    )

print("\n")
print("=" * 60)
print("       SMART NEXT-WORD PREDICTOR")
print("=" * 60)

print("Type a sentence to get Top-5 predictions.")
print("Type 'exit' to stop.\n")


while True:

    sentence = input("Enter sentence: ")

    if sentence.lower().strip() == "exit":

        print("\nPredictor stopped.")
        break


    show_predictions(
        sentence,
        top_n=5
    )

    print()

WikiText-2 dataset loaded successfully!
Total characters: 10780437
Total tokens: 1755457
First 20 tokens:
['valkyria', 'chronicles', 'iii', 'senj', 'no', 'valkyria', '3', 'unk', 'chronicles', 'japanese', '3', 'lit', 'valkyria', 'of', 'the', 'battlefield', '3', 'commonly', 'referred', 'to']

Unique words: 28644
Unique bigrams: 629736
Unique trigrams: 1329996

Input: machine learning is

Top predicted next words:

1. a               Probability: 0.1215
2. the             Probability: 0.0785
3. not             Probability: 0.0298
4. also            Probability: 0.0257
5. an              Probability: 0.0226


TESTING DIFFERENT SENTENCES

Input: machine learning

Top predicted next words:

1. that            Probability: 0.1333
2. curve           Probability: 0.0933
3. the             Probability: 0.0800
4. to              Probability: 0.0800
5. about           Probability: 0.0667

Input: the united

Top predicted next words:

1. states          Probability: 0.7623
2. kingdom         Probab

Enter sentence:  machine learning is



Input: machine learning is

Top predicted next words:

1. a               Probability: 0.1215
2. the             Probability: 0.0785
3. not             Probability: 0.0298
4. also            Probability: 0.0257
5. an              Probability: 0.0226



Enter sentence:  machine learning is



Input: machine learning is

Top predicted next words:

1. a               Probability: 0.1215
2. the             Probability: 0.0785
3. not             Probability: 0.0298
4. also            Probability: 0.0257
5. an              Probability: 0.0226



Enter sentence:  new york



Input: new york

Top predicted next words:

1. times           Probability: 0.1637
2. city            Probability: 0.1457
3. state           Probability: 0.0599
4. in              Probability: 0.0439
5. and             Probability: 0.0399

